In [7]:
!pip -q install stanza

In [8]:
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
import stanza

In [27]:
!wget -q -O data.csv https://raw.githubusercontent.com/IvoDz/lv-text-complexity/refs/heads/main/data/data.csv

In [41]:
df = pd.read_csv("data.csv")
nlp = stanza.Pipeline('lv', processors='tokenize,pos,lemma', use_gpu=False)

INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Loading these models for language: lv (Latvian):
| Processor | Package       |
-----------------------------
| tokenize  | lvtb          |
| pos       | lvtb_nocharlm |
| lemma     | lvtb_nocharlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Done loading processors!


In [54]:
def stanza_tokenizer(text):
    doc = nlp(text)
    tokens = []
    for sentence in doc.sentences:
        for word in sentence.words:
            tokens.append(word.lemma)
    return tokens

In [55]:
le = LabelEncoder()
y_enc = le.fit_transform(df["level"])
X = df["text"]

In [56]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=42
)

# sagatavo tokenizētos datus uzreiz, iekš grid search ir lēnāk, sauc stanza_tokenizer vairākkārt
X_train = [' '.join(stanza_tokenizer(doc)) for doc in X_train_raw]
X_test = [' '.join(stanza_tokenizer(doc)) for doc in X_test_raw]

vectorizer = TfidfVectorizer()

In [60]:
pipeline = Pipeline([
    ('tfidf', vectorizer),
    ('clf', LogisticRegression(max_iter=1000)),
])

param_grid = {
    'tfidf__ngram_range': [(1,1), (1,2), (1,3)],
    'tfidf__min_df': [1, 3],
    'tfidf__max_df': [0.9, 1.0],
    'clf__C': [0.1, 1.0, 10]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='f1_macro',
    verbose=0,
    n_jobs=1,
)

grid.fit(X_train, y_train)

y_pred = grid.predict(X_test)
print("Best params:", grid.best_params_)
print("F1 (macro):", grid.best_score_)
print(classification_report(y_test, y_pred, target_names=le.classes_))

Best params: {'clf__C': 0.1, 'tfidf__max_df': 0.9, 'tfidf__min_df': 3, 'tfidf__ngram_range': (1, 2)}
F1 (macro): 0.47372619738082233
              precision    recall  f1-score   support

   sarežģīts       0.55      0.54      0.54        98
      vidējs       0.40      0.42      0.41       109
      viegls       0.49      0.47      0.48       106

    accuracy                           0.48       313
   macro avg       0.48      0.48      0.48       313
weighted avg       0.48      0.48      0.48       313

